# Phase 2: Exploratory Data Analysis (EDA)

This notebook loads the FORCE 2020 Norwegian Sea well log dataset, visualizes missingness patterns across well curves, plots the class distributions in log scale to observe sample imbalances, visualizes depth-based logs on a multi-track display, and produces statistical summaries.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.data_loader import generate_synthetic_las_files, load_las_dataset

# 1. Initialize folders and data loader
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../plots', exist_ok=True)
generate_synthetic_las_files('../data/raw', num_wells=10)
df = load_las_dataset('../data/raw')
print(f'Merged data shape: {df.shape}')
df.head()

## 1. Visualizing Missing Data

Geophysical well logs frequently contain gaps. We use the `missingno` library to map the missing data density across curves.

In [ ]:
plt.figure(figsize=(12, 6))
msno.matrix(df.drop(columns=['WELL_ID', 'DEPTH_MD', 'LITHOLOGY'], errors='ignore'), sparkline=False)
plt.title('FORCE 2020 Raw Well Logs Missingness Map', fontsize=16)
plt.tight_layout()
plt.savefig('../plots/missingno_matrix.png', dpi=150)
plt.show()

## 2. Lithofacies Class Distribution

Here we plot the class distributions on a logarithmic scale (replicating Figure 4 from the paper) to show class imbalance.

In [ ]:
lithology_labels = {
    0: 'Sandstone', 1: 'Sandstone/Shale', 2: 'Shale', 3: 'Marl', 
    4: 'Dolomite', 5: 'Limestone', 6: 'Chalk', 7: 'Halite', 
    8: 'Anhydrite', 9: 'Tuff', 10: 'Coal', 11: 'Basement'
}

counts = df['LITHOLOGY'].value_counts().sort_index()
labels = [lithology_labels.get(i, f'Class {i}') for i in counts.index]

plt.figure(figsize=(10, 5))
sns.barplot(x=labels, y=counts.values, palette='viridis', hue=labels, legend=False)
plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Lithofacies Class')
plt.ylabel('Sample Count (Log Scale)')
plt.title('FORCE 2020 Lithofacies Class Distribution (Norway Set)', fontsize=14)
plt.grid(axis='y', which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../plots/class_distribution.png', dpi=150)
plt.show()

for i, count in zip(counts.index, counts.values):
    print(f'Class {i:2d} ({lithology_labels[i]:16s}): {count:6d} samples ({count/len(df)*100:5.2f}%)')

## 3. Well Log Track Viewer

A standard geological display plotting key sensor curves (GR, RHOB, NPHI, RMED, DTC) against depth (replicating Figure 3 from the paper).

In [ ]:
well_name = df['WELL_ID'].unique()[0]
well_df = df[df['WELL_ID'] == well_name].sort_values('DEPTH_MD')

fig, axes = plt.subplots(1, 5, figsize=(14, 10), sharey=True)
fig.suptitle(f'Well Log Tracks for Well: {well_name}', fontsize=16, y=1.02)

depth = well_df['DEPTH_MD'].values

# Track 1: Gamma Ray
axes[0].plot(well_df['GR'].values, depth, color='green', lw=1.5)
axes[0].set_xlabel('GR (API)', color='green')
axes[0].set_title('Gamma Ray')
axes[0].grid(True)
axes[0].tick_params(axis='x', labelcolor='green')

# Track 2: Density (RHOB)
axes[1].plot(well_df['RHOB'].values, depth, color='red', lw=1.5)
axes[1].set_xlabel('RHOB (g/cm3)', color='red')
axes[1].set_title('Density')
axes[1].grid(True)
axes[1].tick_params(axis='x', labelcolor='red')

# Track 3: Neutron Porosity (NPHI)
axes[2].plot(well_df['NPHI'].values, depth, color='blue', lw=1.5)
axes[2].set_xlabel('NPHI (v/v)', color='blue')
axes[2].set_title('Neutron Porosity')
axes[2].grid(True)
axes[2].tick_params(axis='x', labelcolor='blue')

# Track 4: Resistivity (RMED)
axes[3].plot(well_df['RMED'].values, depth, color='purple', lw=1.5)
axes[3].set_xscale('log')
axes[3].set_xlabel('RMED (ohm.m)', color='purple')
axes[3].set_title('Resistivity')
axes[3].grid(True)
axes[3].tick_params(axis='x', labelcolor='purple')

# Track 5: Sonic (DTC)
axes[4].plot(well_df['DTC'].values, depth, color='darkorange', lw=1.5)
axes[4].set_xlabel('DTC (us/ft)', color='darkorange')
axes[4].set_title('Sonic Transit Time')
axes[4].grid(True)
axes[4].tick_params(axis='x', labelcolor='darkorange')

# Set vertical axis parameters
axes[0].set_ylabel('Measured Depth (m)', fontsize=12)
axes[0].invert_yaxis()

plt.tight_layout()
plt.savefig(f'../plots/well_tracks_{well_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Statistical Summary

Computes general statistcal descriptors (mean, std, min, max, quartiles) for the 12 selected logs (replicating Table 2 from the paper).

In [ ]:
target_logs = ['DEPTH_MD', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'BS']
summary_stats = df[target_logs].describe().T
summary_stats = summary_stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
summary_stats.columns = ['Mean', 'Std Dev', 'Min', '25% Q', 'Median', '75% Q', 'Max']
print('Well Logs Statistical Summary (Norway Dataset Replicated Table 2):')
summary_stats